# 02 · Computer-Vision Pipeline

Trains and runs the full CV stage end-to-end:

1. **Flower detector** (YOLO26, single class)
2. **Insect detector** (YOLO26, single class) + **pollinator/non-pollinator classifier** (iNaturalist-pretrained)
3. **Tracking + flower-visit counting** on the test videos

Heavy logic lives in `src/cv_engine/`; this notebook orchestrates it. It is
**idempotent** — any model whose weights already exist is loaded, not retrained,
so re-running is safe and fast. Requires the CV environment
(`bash scripts/setup_cv.sh`) and an NVIDIA GPU for training.

In [1]:
import sys, os, json
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

here = Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "src" / "config.py").exists():
        os.chdir(cand); sys.path.insert(0, str(cand)); break
REPO = Path.cwd()
from src import config as C
from src.cv_engine import prepare_flower, prepare_insect, train as cvtrain, train_classifier, visit_counter
import torch
RUNS = C.INTERIM_DIR / "cv_runs"
print("device:", "cuda" if torch.cuda.is_available() else "cpu",
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
def exists(p): return Path(p).exists()

/home/manheim666/Desktop/Bee-A-Hero/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda | NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 1 · Flower detector
Build the single-class flower detection set (GrabCut auto-boxes) and fine-tune YOLO26.

In [2]:
flower_w = RUNS / "flower_yolo26n" / "weights" / "best.pt"
if not exists(flower_w):
    if not exists(C.INTERIM_DIR / "flower_det" / "data.yaml"):
        print(prepare_flower.run(workers=os.cpu_count())["total"], "flower images prepared")
    cvtrain.train(str(C.INTERIM_DIR / "flower_det" / "data.yaml"), "flower_yolo26n", epochs=60)
else:
    print("flower detector already trained — skipping")
print("flower weights:", flower_w)

flower detector already trained — skipping
flower weights: /home/manheim666/Desktop/Bee-A-Hero/data/interim/cv_runs/flower_yolo26n/weights/best.pt


## 2 · Insect detector + pollinator classifier
Build the insect datasets (iNat auto-boxed + BEE.v8i real boxes), train the single-class detector, then the iNaturalist-pretrained classifier.

In [3]:
insect_w = RUNS / "insect1cls_yolo26n" / "weights" / "best.pt"
clf_w = RUNS / "insect_classifier" / "best.pt"

if not exists(C.INTERIM_DIR / "insect_det1"):
    prepare_insect.run(); prepare_insect.export_single_class(); prepare_insect.export_classifier_crops()
    print("insect datasets built")

if not exists(insect_w):
    cvtrain.train(str(C.INTERIM_DIR / "insect_det1" / "data.yaml"), "insect1cls_yolo26n",
                  epochs=60, scale=0.9)
else:
    print("insect detector already trained — skipping")

if not exists(clf_w):
    train_classifier.train(name="insect_classifier", epochs=8, batch=8, freeze=True)
else:
    print("classifier already trained — skipping")
print("insect weights:", insect_w, "| classifier:", clf_w)

insect detector already trained — skipping
classifier already trained — skipping
insect weights: /home/manheim666/Desktop/Bee-A-Hero/data/interim/cv_runs/insect1cls_yolo26n/weights/best.pt | classifier: /home/manheim666/Desktop/Bee-A-Hero/data/interim/cv_runs/insect_classifier/best.pt


## 3 · Metrics

In [4]:
def best_map(csv_path):
    df = pd.read_csv(csv_path); return round(df["metrics/mAP50(B)"].max(), 4)
metrics = {
    "flower_detector_mAP50": best_map(RUNS / "flower_yolo26n" / "results.csv"),
    "insect_detector_mAP50": best_map(RUNS / "insect1cls_yolo26n" / "results.csv"),
    "classifier": json.load(open(RUNS / "insect_classifier" / "classifier_result.json"))["test"],
}
print(json.dumps(metrics, indent=2))

{
  "flower_detector_mAP50": 0.9176,
  "insect_detector_mAP50": 0.9002,
  "classifier": {
    "acc": 0.9783,
    "balanced_acc": 0.9778,
    "pollinator_recall": 0.9749,
    "non_pollinator_recall": 0.9806
  }
}


## 4 · Tracking + flower-visit counting
Run every test video through detect → track → classify → count. Flowers are detected per-frame with stable IDs; an insect entering a flower ROI is one visit.

In [5]:
out_dir = RUNS / "visits"
videos = sorted((C.TEST_VIDEO_DIR).glob("*.mp4"))
results = []
for v in videos:
    r = visit_counter.count_visits(str(v), str(flower_w), str(insect_w), str(clf_w),
                                   out_dir, save_video=True, flower_interval=5)
    results.append(r); print(v.name, "->", r["flowers"], "flowers,", sum(r["visits"].values()), "visits")

videoplayback (1).mp4 -> 2 flowers, 11 visits


videoplayback (2).mp4 -> 1 flowers, 103 visits


videoplayback (3).mp4 -> 2 flowers, 16 visits


videoplayback (4).mp4 -> 1 flowers, 10 visits


videoplayback.mp4 -> 1 flowers, 8 visits


### Visit tallies

In [6]:
frames = []
for csvf in sorted(out_dir.glob("*_visits.csv")):
    d = pd.read_csv(csvf); d.insert(0, "video", csvf.stem.replace("_visits", ""))
    frames.append(d)
visits_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
visits_df[visits_df["total"] > 0] if not visits_df.empty else visits_df

,video,flower_id,total,pollinator,non_pollinator
0,videoplayback (1),flower_1,10,6,4
4,videoplayback (1),flower_5,1,1,0
5,videoplayback (2),flower_1,15,15,0
7,videoplayback (2),flower_11,14,0,14
10,videoplayback (2),flower_14,1,0,1
12,videoplayback (2),flower_16,13,3,10
14,videoplayback (2),flower_18,3,0,3
16,videoplayback (2),flower_2,4,4,0
17,videoplayback (2),flower_3,1,1,0
18,videoplayback (2),flower_4,48,33,15


### Sample annotated frame

In [7]:
import cv2, glob
vids = sorted(glob.glob(str(out_dir / "*_annotated.mp4")), key=os.path.getsize)
if vids:
    cap = cv2.VideoCapture(vids[-1]); cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))
    ok, fr = cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(11, 6)); plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); plt.axis("off")
        plt.title("flowers (green) + insect tracks (bee=orange / non=red)"); plt.show()

/tmp/ipykernel_126970/2060995824.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("flowers (green) + insect tracks (bee=orange / non=red)"); plt.show()


## 5 · Summary
All trained weights are under `data/interim/cv_runs/*/weights/`; annotated videos + visit CSVs under `data/interim/cv_runs/visits/`.

In [8]:
print(json.dumps({"metrics": metrics,
                  "videos_processed": len(results),
                  "outputs": str(out_dir)}, indent=2))

{
  "metrics": {
    "flower_detector_mAP50": 0.9176,
    "insect_detector_mAP50": 0.9002,
    "classifier": {
      "acc": 0.9783,
      "balanced_acc": 0.9778,
      "pollinator_recall": 0.9749,
      "non_pollinator_recall": 0.9806
    }
  },
  "videos_processed": 5,
  "outputs": "/home/manheim666/Desktop/Bee-A-Hero/data/interim/cv_runs/visits"
}
